# Lahore POIs (Fetch Only)

Fetch and save **all POIs for Lahore** into local GeoJSON/CSV.

Default source: **Overture Places** (more complete and stable than Overpass).


In [28]:
import geopandas as gpd
import pandas as pd
import os


In [29]:
# ---- CONFIG ----
BOUNDARY_PATH = "../../data/Lahore UCs/Lahore UC.shp"

# Output
OUT_GEOJSON = "./data/pois_lahore.geojson"
OUT_CSV = "./data/pois_lahore.csv"

# Source: "overture" (recommended) or "osmnx"
POI_SOURCE = "overture"

# Overture release (latest from https://labs.overturemaps.org/data/releases.json)
OVERTURE_RELEASE = "2026-01-21.0"
OVERTURE_BASE = f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=places/type=place/*"

# OSMnx fallback (less complete, rate-limited)
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter",
]
SIMPLIFY_TOL_DEG = 0.002


In [30]:
# ---- LOAD BOUNDARY ----
boundary = gpd.read_file(BOUNDARY_PATH).to_crs(4326)
lahore_boundary = boundary.unary_union
print("Boundary loaded")


Boundary loaded


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_63394/1831077039.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  lahore_boundary = boundary.unary_union


In [31]:
# ---- FETCH POIS: OVERTURE ----

def fetch_overture_pois(boundary_polygon):
    import duckdb
    import shapely.wkb

    minx, miny, maxx, maxy = boundary.total_bounds

    con = duckdb.connect()
    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")
    con.execute("INSTALL spatial")
    con.execute("LOAD spatial")
    con.execute("SET s3_region='us-west-2'")

    # Inspect schema to detect bbox column if available
    schema = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{OVERTURE_BASE}') LIMIT 1"
    ).fetchall()
    cols = [r[0] for r in schema]

    if "bbox" in cols:
        bbox_pred = (
            f"bbox.xmin <= {maxx} AND bbox.xmax >= {minx} "
            f"AND bbox.ymin <= {maxy} AND bbox.ymax >= {miny}"
        )
    else:
        bbox_pred = "1=1"

    query = (
        "SELECT id, names, categories, taxonomy, basic_category, confidence, operating_status, "
        "ST_AsWKB(geometry) AS geometry_wkb "
        f"FROM read_parquet('{OVERTURE_BASE}', filename=true, hive_partitioning=1) "
        f"WHERE {bbox_pred}"
    )

    df = con.execute(query).df()
    con.close()

    # Convert WKB to geometry (bytes only)
    geom_raw = df["geometry_wkb"]
    geom_raw = geom_raw.apply(lambda x: bytes(x) if isinstance(x, bytearray) else x)
    mask = geom_raw.apply(lambda x: isinstance(x, (bytes, bytearray)))
    df = df.loc[mask].copy()
    geom_raw = geom_raw[mask]

    df["geometry"] = gpd.GeoSeries.from_wkb(geom_raw)
    df = df.drop(columns=["geometry_wkb"])

    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=4326)

    # Clip to Lahore boundary
    gdf = gdf[gdf.geometry.within(boundary_polygon)]
    return gdf


In [32]:
# ---- FETCH POIS: OSMNX (fallback) ----
import osmnx as ox

def _query_polygon(tags, poly):
    if hasattr(ox, "geometries_from_polygon"):
        return ox.geometries_from_polygon(poly, tags)
    return ox.features_from_polygon(poly, tags)


def fetch_osmnx_pois(boundary_polygon):
    poly = boundary_polygon
    try:
        poly = poly.buffer(0).simplify(SIMPLIFY_TOL_DEG)
    except Exception:
        pass

    # broad tags: all common POI-bearing keys
    tags = {
        "amenity": True,
        "shop": True,
        "tourism": True,
        "leisure": True,
        "office": True,
        "craft": True,
        "man_made": True,
        "public_transport": True,
        "railway": True,
    }

    last_err = None
    for url in OVERPASS_URLS:
        try:
            ox.settings.overpass_url = url
            gdf = _query_polygon(tags, poly)
            if gdf is not None and len(gdf) > 0:
                gdf = gdf.reset_index(drop=True)
                # geometry to points
                gdf["geometry"] = gdf.geometry.centroid
                gdf = gdf[~gdf.geometry.is_empty].copy()
                return gdf
        except Exception as e:
            last_err = e
            print("Overpass failed:", url, "->", e)

    print("OSMnx returned no data.")
    if last_err:
        print("Last error:", last_err)

    return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=4326)


In [33]:
# ---- RUN + SAVE ----
os.makedirs("./data", exist_ok=True)

if POI_SOURCE == "overture":
    pois = fetch_overture_pois(lahore_boundary)
elif POI_SOURCE == "osmnx":
    pois = fetch_osmnx_pois(lahore_boundary)
else:
    raise ValueError("Unknown POI_SOURCE")

print("POIs:", len(pois))

pois.to_file(OUT_GEOJSON, driver="GeoJSON")
pois.drop(columns="geometry").to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_GEOJSON, OUT_CSV)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

POIs: 36807
Wrote: ./data/pois_lahore.geojson ./data/pois_lahore.csv


In [34]:
# ---- GROUP POIS BY TAXONOMY ----
# Use Overture taxonomy.hierarchy for higher-level grouping.

import json as _json

# Expect taxonomy as dict-like in "taxonomy" column
if "taxonomy" not in pois.columns:
    raise ValueError("taxonomy not found in POIs. Check Overture schema.")

# Extract top-level taxonomy (hierarchy[0]) if available

def _top_taxonomy(val):
    if val is None:
        return "unknown"
    # If stored as dict
    if isinstance(val, dict):
        h = val.get("hierarchy")
        if isinstance(h, (list, tuple)) and len(h) > 0:
            return h[0]
        return val.get("primary", "unknown")
    # If stored as JSON string
    if isinstance(val, str):
        try:
            obj = _json.loads(val)
            h = obj.get("hierarchy") if isinstance(obj, dict) else None
            if isinstance(h, (list, tuple)) and len(h) > 0:
                return h[0]
            if isinstance(obj, dict) and "primary" in obj:
                return obj.get("primary")
        except Exception:
            return "unknown"
    return "unknown"

pois["group"] = pois["taxonomy"].apply(_top_taxonomy)

# Save grouped POIs
OUT_GROUPED_GEOJSON = "./data/pois_lahore_grouped.geojson"
OUT_GROUPED_CSV = "./data/pois_lahore_grouped.csv"

pois.to_file(OUT_GROUPED_GEOJSON, driver="GeoJSON")
pois.drop(columns="geometry").to_csv(OUT_GROUPED_CSV, index=False)

print(pois["group"].value_counts())
print("Wrote:", OUT_GROUPED_GEOJSON, OUT_GROUPED_CSV)


group
services_and_business        12299
shopping                      6717
food_and_drink                4366
education                     3621
travel_and_transportation     2136
health_care                   2078
community_and_government      1250
lifestyle_services            1238
cultural_and_historic          903
lodging                        903
sports_and_recreation          873
arts_and_entertainment         387
geographic_entities             36
Name: count, dtype: int64
Wrote: ./data/pois_lahore_grouped.geojson ./data/pois_lahore_grouped.csv


In [35]:
# ---- PLOT POIS (TAXONOMY COLOR) ----
import folium

# Center map
cent = boundary.to_crs(4326).geometry.unary_union.centroid
m = folium.Map(location=[cent.y, cent.x], zoom_start=11, tiles="cartodbpositron")

# Boundary
folium.GeoJson(boundary.to_crs(4326)).add_to(m)

# Color palette for top-level taxonomy (fallback to gray)
GROUP_COLORS = {
    "services_and_business": "#8c510a",
    "shopping": "#01665e",
    "food_and_drink": "#bf812d",
    "health_care": "#b2182b",
    "travel_and_transportation": "#2166ac",
    "lifestyle_services": "#5ab4ac",
    "education": "#762a83",
    "community_and_government": "#1b7837",
    "cultural_and_historic": "#d8b365",
    "sports_and_recreation": "#f46d43",
    "lodging": "#4d4d4d",
    "arts_and_entertainment": "#66c2a5",
    "geographic_entities": "#999999",
    "unknown": "#cccccc",
}

# Plot POIs
if not pois.empty:
    for _, row in pois.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        name = None
        if "names" in row and row["names"] is not None:
            try:
                name = row["names"].get("primary")
            except Exception:
                name = None
        label = name if name else row.get("basic_category", "POI")
        group = row.get("group", "unknown")
        color = GROUP_COLORS.get(group, "#cccccc")

        folium.CircleMarker(
            location=[geom.y, geom.x],
            radius=2,
            color=color,
            fill=True,
            fill_opacity=0.8,
            tooltip=f"{group}: {label}",
        ).add_to(m)

OUT_MAP = "./lahore_pois_map.html"
m.save(OUT_MAP)
print("Wrote:", OUT_MAP)

#m


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_63394/584316900.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cent = boundary.to_crs(4326).geometry.unary_union.centroid


Wrote: ./lahore_pois_map.html
